In [1]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_PATH = Path("../DATA/raw/CERT_R6.2/data")

In [3]:
# we are working with logon and logoff data, so we will load the logon.csv file.
logon = pd.read_csv(DATA_PATH / "logon.csv")

In [4]:
logon.head()

,id,date,user,pc,activity
0,{F3X8-Y2GT43DR-4906OHBL},01/02/2010 02:19:18,DNS1758,PC-0414,Logon
1,{B4Q0-D0GM24KN-3704MAII},01/02/2010 02:31:12,DNS1758,PC-0414,Logoff
2,{T7J1-D4HK34KV-5476TCIJ},01/02/2010 02:34:02,DNS1758,PC-5313,Logon
3,{S4Y6-D8MQ05SA-0759HLIS},01/02/2010 02:53:30,DNS1758,PC-5313,Logoff
4,{F3P0-E7FH78CV-4874FRGZ},01/02/2010 04:07:31,DNS1758,PC-0012,Logon


In [5]:
logon.tail()

,id,date,user,pc,activity
3530280,{D2Y5-B9RW00TS-7496JXWF},06/01/2011 05:39:10,QAH1315,PC-3767,Logoff
3530281,{O2Y5-U5OP72ZC-4270UMRB},06/01/2011 06:07:37,HMY0235,PC-7084,Logoff
3530282,{L0C2-R9VH68RL-6266MDMB},06/01/2011 06:13:11,CRM0139,PC-8757,Logoff
3530283,{C4W5-C1PN84LC-1265PUOM},06/01/2011 06:49:27,ABM3641,PC-7385,Logoff
3530284,{U5Y4-A0BC08BH-2666UMVL},06/01/2011 07:26:02,KCH2475,PC-4356,Logoff


In [6]:
logon.shape

(3530285, 5)

In [7]:
logon.columns

Index(['id', 'date', 'user', 'pc', 'activity'], dtype='str')

In [8]:
logon.info()

<class 'pandas.DataFrame'>
RangeIndex: 3530285 entries, 0 to 3530284
Data columns (total 5 columns):
 #   Column    Dtype
---  ------    -----
 0   id        str  
 1   date      str  
 2   user      str  
 3   pc        str  
 4   activity  str  
dtypes: str(5)
memory usage: 344.9 MB


In [9]:
logon.describe(include="all")

,id,date,user,pc,activity
count,3530285,3530285,3530285,3530285,3530285
unique,3530285,955571,4000,4400,2
top,{F3X8-Y2GT43DR-4906OHBL},01/07/2010 08:00:00,TAM3048,PC-6793,Logon
freq,1,199,4151,1654,1948933


In [10]:
# Check Missing Values
logon.isnull().sum()

id          0
date        0
user        0
pc          0
activity    0
dtype: int64

In [11]:
# Check Duplicate Records
logon.duplicated().sum()

np.int64(0)

In [12]:
# Check Unique Users
logon["user"].nunique()

4000

In [13]:
# Check Activity Types
logon["activity"].value_counts()

activity
Logon     1948933
Logoff    1581352
Name: count, dtype: int64

In [14]:
# Convert Date Column
logon["date"] = pd.to_datetime(logon["date"])


In [15]:
# Verify Datatype
logon.info()

<class 'pandas.DataFrame'>
RangeIndex: 3530285 entries, 0 to 3530284
Data columns (total 5 columns):
 #   Column    Dtype         
---  ------    -----         
 0   id        str           
 1   date      datetime64[us]
 2   user      str           
 3   pc        str           
 4   activity  str           
dtypes: datetime64[us](1), str(4)
memory usage: 280.9 MB


In [16]:
# Extract Date Features
logon["year"] = logon["date"].dt.year
logon["month"] = logon["date"].dt.month
logon["day"] = logon["date"].dt.day
logon["hour"] = logon["date"].dt.hour
logon["weekday"] = logon["date"].dt.day_name()

In [17]:
# Display Updated Dataset
logon.head()

,id,date,user,pc,activity,year,month,day,hour,weekday
0,{F3X8-Y2GT43DR-4906OHBL},2010-01-02 02:19:18,DNS1758,PC-0414,Logon,2010,1,2,2,Saturday
1,{B4Q0-D0GM24KN-3704MAII},2010-01-02 02:31:12,DNS1758,PC-0414,Logoff,2010,1,2,2,Saturday
2,{T7J1-D4HK34KV-5476TCIJ},2010-01-02 02:34:02,DNS1758,PC-5313,Logon,2010,1,2,2,Saturday
3,{S4Y6-D8MQ05SA-0759HLIS},2010-01-02 02:53:30,DNS1758,PC-5313,Logoff,2010,1,2,2,Saturday
4,{F3P0-E7FH78CV-4874FRGZ},2010-01-02 04:07:31,DNS1758,PC-0012,Logon,2010,1,2,4,Saturday


In [18]:
# Create After Hours Feature
logon["after_hours"] = (
    (logon["hour"] < 8) |
    (logon["hour"] >= 18)
)

In [19]:
# Create Weekend Feature
logon["weekend_login"] = logon["weekday"].isin(
    ["Saturday", "Sunday"]
)

In [20]:
# Create Employee Profile
user_profile = (
    logon.groupby("user")
    .size()
    .reset_index(name="total_events")
)

user_profile.head()

,user,total_events
0,AAB0162,711
1,AAB0398,712
2,AAC0610,1052
3,AAC0668,712
4,AAC3270,712


In [21]:
# Calculate Total Logons
total_logons = (
    logon[logon["activity"] == "Logon"]
    .groupby("user")
    .size()
    .reset_index(name="total_logons")
)

total_logons.head()

,user,total_logons
0,AAB0162,355
1,AAB0398,356
2,AAC0610,665
3,AAC0668,356
4,AAC3270,356


In [22]:
# Merge Total Logons
user_profile = user_profile.merge(
    total_logons,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons
0,AAB0162,711,355
1,AAB0398,712,356
2,AAC0610,1052,665
3,AAC0668,712,356
4,AAC3270,712,356


In [23]:
# Calculate Total Logoffs
total_logoffs = (
    logon[logon["activity"] == "Logoff"]
    .groupby("user")
    .size()
    .reset_index(name="total_logoffs")
)

total_logoffs.head()

,user,total_logoffs
0,AAB0162,356
1,AAB0398,356
2,AAC0610,387
3,AAC0668,356
4,AAC3270,356


In [24]:
# Merge Total Logoffs
user_profile = user_profile.merge(
    total_logoffs,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons,total_logoffs
0,AAB0162,711,355,356
1,AAB0398,712,356,356
2,AAC0610,1052,665,387
3,AAC0668,712,356,356
4,AAC3270,712,356,356


In [25]:
# Calculate After Hours Events
after_hours_events = (
    logon[logon["after_hours"]]
    .groupby("user")
    .size()
    .reset_index(name="after_hours_events")
)

after_hours_events.head()

,user,after_hours_events
0,AAB0162,705
1,AAB0398,356
2,AAC0610,222
3,AAC0668,158
4,AAD2188,356


In [26]:
# Merge After Hours Events
user_profile = user_profile.merge(
    after_hours_events,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events
0,AAB0162,711,355,356,705.0
1,AAB0398,712,356,356,356.0
2,AAC0610,1052,665,387,222.0
3,AAC0668,712,356,356,158.0
4,AAC3270,712,356,356,NaN


In [27]:
# Calculate Weekend Events
weekend_events = (
    logon[logon["weekend_login"]]
    .groupby("user")
    .size()
    .reset_index(name="weekend_events")
)

weekend_events.head()

,user,weekend_events
0,AAC0610,5
1,AAS3428,228
2,ABB0300,44
3,ABD3426,323
4,ABK3081,201


In [28]:
# Merge Weekend Events
user_profile = user_profile.merge(
    weekend_events,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events
0,AAB0162,711,355,356,705.0,NaN
1,AAB0398,712,356,356,356.0,NaN
2,AAC0610,1052,665,387,222.0,5.0
3,AAC0668,712,356,356,158.0,NaN
4,AAC3270,712,356,356,NaN,NaN


In [29]:
# Calculate Unique PCs Used
unique_pcs = (
    logon.groupby("user")["pc"]
    .nunique()
    .reset_index(name="unique_pcs")
)

unique_pcs.head()

,user,unique_pcs
0,AAB0162,1
1,AAB0398,1
2,AAC0610,1
3,AAC0668,1
4,AAC3270,1


In [30]:
# Merge Unique PCs
user_profile = user_profile.merge(
    unique_pcs,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs
0,AAB0162,711,355,356,705.0,NaN,1
1,AAB0398,712,356,356,356.0,NaN,1
2,AAC0610,1052,665,387,222.0,5.0,1
3,AAC0668,712,356,356,158.0,NaN,1
4,AAC3270,712,356,356,NaN,NaN,1


In [31]:
# Calculate Working Days
working_days = (
    logon.groupby("user")["date"]
    .apply(lambda x: x.dt.date.nunique())
    .reset_index(name="working_days")
)

working_days.head()

,user,working_days
0,AAB0162,356
1,AAB0398,356
2,AAC0610,360
3,AAC0668,356
4,AAC3270,356


In [32]:
# Merge Working Days
user_profile = user_profile.merge(
    working_days,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days
0,AAB0162,711,355,356,705.0,NaN,1,356
1,AAB0398,712,356,356,356.0,NaN,1,356
2,AAC0610,1052,665,387,222.0,5.0,1,360
3,AAC0668,712,356,356,158.0,NaN,1,356
4,AAC3270,712,356,356,NaN,NaN,1,356


In [33]:
# Calculate Average Events Per Day
user_profile["avg_events_per_day"] = (
    user_profile["total_events"] /
    user_profile["working_days"]
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days,avg_events_per_day
0,AAB0162,711,355,356,705.0,NaN,1,356,1.997191
1,AAB0398,712,356,356,356.0,NaN,1,356,2.000000
2,AAC0610,1052,665,387,222.0,5.0,1,360,2.922222
3,AAC0668,712,356,356,158.0,NaN,1,356,2.000000
4,AAC3270,712,356,356,NaN,NaN,1,356,2.000000


In [34]:
# Calculate First Login Hour
first_login = (
    logon[logon["activity"] == "Logon"]
    .groupby("user")["hour"]
    .min()
    .reset_index(name="first_login_hour")
)

first_login.head()

,user,first_login_hour
0,AAB0162,7
1,AAB0398,6
2,AAC0610,0
3,AAC0668,7
4,AAC3270,8


In [35]:
# Merge First Login Hour
user_profile = user_profile.merge(
    first_login,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days,avg_events_per_day,first_login_hour
0,AAB0162,711,355,356,705.0,NaN,1,356,1.997191,7
1,AAB0398,712,356,356,356.0,NaN,1,356,2.000000,6
2,AAC0610,1052,665,387,222.0,5.0,1,360,2.922222,0
3,AAC0668,712,356,356,158.0,NaN,1,356,2.000000,7
4,AAC3270,712,356,356,NaN,NaN,1,356,2.000000,8


In [36]:
# Calculate Last Logout Hour
last_logout = (
    logon[logon["activity"] == "Logoff"]
    .groupby("user")["hour"]
    .max()
    .reset_index(name="last_logout_hour")
)

last_logout.head()

,user,last_logout_hour
0,AAB0162,19
1,AAB0398,16
2,AAC0610,17
3,AAC0668,17
4,AAC3270,16


In [37]:
# Merge Last Logout Hour
user_profile = user_profile.merge(
    last_logout,
    on="user",
    how="left"
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days,avg_events_per_day,first_login_hour,last_logout_hour
0,AAB0162,711,355,356,705.0,NaN,1,356,1.997191,7,19
1,AAB0398,712,356,356,356.0,NaN,1,356,2.000000,6,16
2,AAC0610,1052,665,387,222.0,5.0,1,360,2.922222,0,17
3,AAC0668,712,356,356,158.0,NaN,1,356,2.000000,7,17
4,AAC3270,712,356,356,NaN,NaN,1,356,2.000000,8,16


In [38]:
# Calculate Estimated Work Duration
user_profile["work_duration_hours"] = (
    user_profile["last_logout_hour"] -
    user_profile["first_login_hour"]
)

user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days,avg_events_per_day,first_login_hour,last_logout_hour,work_duration_hours
0,AAB0162,711,355,356,705.0,NaN,1,356,1.997191,7,19,12
1,AAB0398,712,356,356,356.0,NaN,1,356,2.000000,6,16,10
2,AAC0610,1052,665,387,222.0,5.0,1,360,2.922222,0,17,17
3,AAC0668,712,356,356,158.0,NaN,1,356,2.000000,7,17,10
4,AAC3270,712,356,356,NaN,NaN,1,356,2.000000,8,16,8


In [39]:
# Fill Missing Values
user_profile = user_profile.fillna(0)

In [40]:
# Check Dataset Information
user_profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user                 4000 non-null   str    
 1   total_events         4000 non-null   int64  
 2   total_logons         4000 non-null   int64  
 3   total_logoffs        4000 non-null   int64  
 4   after_hours_events   4000 non-null   float64
 5   weekend_events       4000 non-null   float64
 6   unique_pcs           4000 non-null   int64  
 7   working_days         4000 non-null   int64  
 8   avg_events_per_day   4000 non-null   float64
 9   first_login_hour     4000 non-null   int32  
 10  last_logout_hour     4000 non-null   int32  
 11  work_duration_hours  4000 non-null   int32  
dtypes: float64(3), int32(3), int64(5), str(1)
memory usage: 356.1 KB


In [41]:
# Display First Five Rows
user_profile.head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days,avg_events_per_day,first_login_hour,last_logout_hour,work_duration_hours
0,AAB0162,711,355,356,705.0,0.0,1,356,1.997191,7,19,12
1,AAB0398,712,356,356,356.0,0.0,1,356,2.000000,6,16,10
2,AAC0610,1052,665,387,222.0,5.0,1,360,2.922222,0,17,17
3,AAC0668,712,356,356,158.0,0.0,1,356,2.000000,7,17,10
4,AAC3270,712,356,356,0.0,0.0,1,356,2.000000,8,16,8


In [42]:
# Display Statistical Summary
user_profile.describe()

,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days,avg_events_per_day,first_login_hour,last_logout_hour,work_duration_hours
count,4000.000000,4000.00000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,882.571250,487.23325,395.338000,345.264250,20.054500,38.791000,348.502500,2.527725,6.441000,17.926000,11.485000
std,411.251398,245.38082,182.557717,329.481241,72.046594,144.937135,62.842571,1.063629,2.435673,2.118879,4.278183
min,42.000000,21.00000,21.000000,0.000000,0.000000,1.000000,21.000000,1.994382,0.000000,14.000000,7.000000
25%,712.000000,356.00000,356.000000,166.000000,0.000000,1.000000,356.000000,2.000000,7.000000,17.000000,9.000000
50%,712.000000,356.00000,356.000000,356.000000,0.000000,1.000000,356.000000,2.000000,7.000000,18.000000,10.000000
75%,996.000000,637.00000,395.000000,373.000000,0.000000,19.000000,356.000000,2.800112,8.000000,18.000000,11.000000
max,4151.000000,2256.00000,1895.000000,2763.000000,1002.000000,1226.000000,515.000000,9.794749,9.000000,23.000000,23.000000


In [43]:
# Save Engineered Employee Profile
FEATURE_PATH = Path("../DATA/features")
FEATURE_PATH.mkdir(parents=True, exist_ok=True)

user_profile.to_csv(
    FEATURE_PATH / "employee_profile.csv",
    index=False
)

print("employee_profile.csv saved successfully!")

employee_profile.csv saved successfully!


In [44]:
# Verify Saved File
pd.read_csv(FEATURE_PATH / "employee_profile.csv").head()

,user,total_events,total_logons,total_logoffs,after_hours_events,weekend_events,unique_pcs,working_days,avg_events_per_day,first_login_hour,last_logout_hour,work_duration_hours
0,AAB0162,711,355,356,705.0,0.0,1,356,1.997191,7,19,12
1,AAB0398,712,356,356,356.0,0.0,1,356,2.000000,6,16,10
2,AAC0610,1052,665,387,222.0,5.0,1,360,2.922222,0,17,17
3,AAC0668,712,356,356,158.0,0.0,1,356,2.000000,7,17,10
4,AAC3270,712,356,356,0.0,0.0,1,356,2.000000,8,16,8


In [45]:
# Final Employee Profile Shape
user_profile.shape

(4000, 12)